In [ ]:
# nb_bi_refresh — rebuild the BI marts after gold is written.
#
# fact_eod_sale_product -> vw_eod_sales / vw_eod_sales_daily (views)
#   -> bi_eod_sales / bi_eod_sales_daily (Delta CTAS, Direct Lake reads these)
#   -> bi_dim_product (CURRENT slice of the SCD2 silver.dim_product + brand/mfr)
#   -> gold.dim_date (calendar spanning the fact's date range)
#
# Runs as the LAST activity of Pipeline_eod_sale_product (after nb_dq_check),
# so dashboards refresh only from data that passed the DQ gate.
# Full rebuild each run: marts are small aggregates, CREATE OR REPLACE is atomic.

In [ ]:
%%sql
CREATE OR REPLACE VIEW gold.vw_eod_sales AS
SELECT
    f.report_date,
    year(f.report_date)                                   AS year,
    date_format(f.report_date, 'yyyy-MM')                 AS month_key,
    date_trunc('WEEK', f.report_date)                     AS week_start,   -- Monday
    date_format(f.report_date, 'EEE')                     AS day_name,
    CASE WHEN dayofweek(f.report_date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,  -- 1=Sun,7=Sat
    hour(f.transaction_time)                              AS txn_hour,
    CASE
        WHEN hour(f.transaction_time) BETWEEN 6  AND 10 THEN '1 Morning (06-10)'
        WHEN hour(f.transaction_time) BETWEEN 11 AND 13 THEN '2 Lunch (11-13)'
        WHEN hour(f.transaction_time) BETWEEN 14 AND 16 THEN '3 Afternoon (14-16)'
        WHEN hour(f.transaction_time) BETWEEN 17 AND 21 THEN '4 Evening (17-21)'
        ELSE '5 Night (22-05)'
    END                                                   AS daypart,
    CASE
        WHEN nullif(f.transaction_7now_source, '') IS NOT NULL       THEN '7NOW'
        WHEN upper(coalesce(f.channel_name, '')) LIKE '%SAVYU%'      THEN 'Savyu'
        WHEN upper(coalesce(f.channel_name, '')) LIKE '%7NOW%'       THEN '7NOW'
        WHEN nullif(f.channel_name, '') IS NOT NULL                  THEN f.channel_name
        ELSE 'Store'
    END                                                   AS channel,
    f.store_id,
    f.store_name,
    f.store_type,
    f.transaction_id,
    f.product_id,
    f.quantity,
    f.final_amount_no_tax                                 AS revenue_net,
    f.final_amount                                        AS revenue_gross,
    f.customer_code,
    CASE WHEN nullif(f.customer_code, '') IS NOT NULL THEN 1 ELSE 0 END AS is_identified_customer,
    -- Customer analytics (labels decoded in silver; NULLIF '' -> cleaner slicers)
    f.customer_id,
    nullif(f.customer_gender, '')                         AS customer_gender,
    nullif(f.customer_age_range, '')                      AS customer_age_range,
    nullif(f.customer_nationality, '')                    AS customer_nationality,
    f.rating,
    -- Margin: unit cost from the effective-dated purchase-price timeline (joined in gold)
    f.purchase_price_without_tax * f.quantity             AS cogs
FROM gold.fact_eod_sale_product f
WHERE f.delivery_status = 'completed'
  AND f.transaction_type = 'Sale Transaction'
  AND f.store_id NOT IN ('1000', '1004')
  AND f.store_id NOT LIKE '19%';

In [ ]:
%%sql
CREATE OR REPLACE VIEW gold.vw_eod_sales_daily AS
SELECT
    report_date,
    year,
    month_key,
    week_start,
    is_weekend,
    channel,
    count(DISTINCT transaction_id)                        AS baskets,
    count(DISTINCT store_id)                              AS stores,
    sum(quantity)                                         AS units,
    round(sum(revenue_net), 0)                            AS revenue_net,
    round(sum(revenue_gross), 0)                          AS revenue_gross,
    round(sum(revenue_net) / nullif(count(DISTINCT transaction_id), 0), 0) AS basket_size
FROM gold.vw_eod_sales
GROUP BY report_date, year, month_key, week_start, is_weekend, channel;

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.bi_eod_sales       USING DELTA AS SELECT * FROM gold.vw_eod_sales;
CREATE OR REPLACE TABLE gold.bi_eod_sales_daily USING DELTA AS SELECT * FROM gold.vw_eod_sales_daily;
-- Direct Lake-safe CURRENT view of the SCD2 product dim (1 row per product_id).
-- The semantic model must never point at silver.dim_product directly: SCD2 versions
-- would break the 1:* relationship the first time a product changes.
CREATE OR REPLACE TABLE gold.bi_dim_product     USING DELTA AS
SELECT p.product_id,
       p.product_name,
       p.product_category_name     AS product_category,
       p.product_group_name        AS product_group,
       p.product_sub_category_name AS product_sub_category,
       p.retail_business_type,
       o.product_brand_name        AS brand,
       o.product_manufacturer_name AS manufacturer
FROM silver.dim_product p
LEFT JOIN silver.dim_oms_product o USING (product_id)
WHERE p.is_current;

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT
    d                                            AS date,
    year(d)                                      AS year,
    date_format(d, 'yyyy-MM')                    AS month_key,
    date_format(d, 'MMM yyyy')                   AS month,
    date_trunc('WEEK', d)                        AS week_start,        -- Monday
    date_format(d, 'EEE')                        AS day_name,
    ((dayofweek(d) + 5) % 7) + 1                 AS weekday_no,        -- 1=Mon..7=Sun (để sort day_name)
    CASE WHEN dayofweek(d) IN (1,7) THEN true ELSE false END AS is_weekend,
    CASE WHEN dayofweek(d) IN (1,7) THEN 'Weekend' ELSE 'Weekday' END AS day_type   -- legend-friendly label
FROM (
    SELECT explode(sequence(
        (SELECT min(report_date) FROM gold.bi_eod_sales),
        (SELECT max(report_date) FROM gold.bi_eod_sales),
        interval 1 day)) AS d
);